# 03 — Talkbot: Voice Coaching Interface

**Purpose**: Install and configure [talkbot](https://github.com/ricklon/talkbot) on the Pi as a local-first voice assistant for coaching sessions. Students speak to the robot setup — narrate episodes, ask status questions, receive verbal feedback — while physically coaching the arm.

**Prereqs**:
- `.env` populated with `PI_HOST`, `PI_PORT`, and optionally `OPENROUTER_API_KEY`
- Pi reachable via SSH
- Microphone and speaker accessible to the Pi (USB audio or 3.5mm)

**Outcome**: Talkbot running on the Pi in a tmux session with a robot coaching prompt. Students can use talkbot alone or alongside LeRobot (see `04_lerobot_talkbot.ipynb`).

---

In [ ]:
import os
import shlex
import time
import json
import subprocess
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv
from tqdm.notebook import tqdm

load_dotenv(dotenv_path=Path('..') / '.env', override=False)

# Benchmark: record start time
_bench = {
    "notebook": "03_talkbot",
    "started_at": datetime.utcnow().isoformat(),
    "timings": {},
    "config": {}
}
_t0 = time.monotonic()

PI_HOST            = os.getenv("PI_HOST", "192.168.4.191")
PI_PORT            = os.getenv("PI_PORT", "22222")
PI_USER            = "root"
LLM_BACKEND        = os.getenv("TALKBOT_LLM_BACKEND", "local")
AGENT_PROMPT       = os.getenv(
    "TALKBOT_AGENT_PROMPT",
    "You are a coaching assistant for a robot arm. "
    "Help the student record quality demonstration episodes. "
    "Keep responses brief — the student is actively coaching."
)
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "")

_bench["config"] = {"pi": PI_HOST, "llm_backend": LLM_BACKEND}

def pi_run(cmd, capture=False, input_data=None, stream=False):
    """Run a command on the Pi over SSH."""
    ssh_prefix = [
        "ssh", "-p", PI_PORT,
        "-o", "StrictHostKeyChecking=no",
        "-o", "ConnectTimeout=10",
        f"{PI_USER}@{PI_HOST}",
    ]
    full = ssh_prefix + ["bash", "-c", cmd]
    if stream:
        proc = subprocess.Popen(full, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in proc.stdout:
            print(line, end="")
        return proc.wait()
    elif capture:
        return subprocess.check_output(full, text=True).strip()
    else:
        return subprocess.run(full, input=input_data, capture_output=True, text=True)

print(f"Pi:          {PI_HOST}:{PI_PORT}")
print(f"LLM backend: {LLM_BACKEND}")
print(f"Agent prompt: {AGENT_PROMPT[:80]}...")

## 1. Verify Pi Connectivity

In [ ]:
_t_connect = time.monotonic()

result = pi_run("echo ok && uname -m && cat /etc/os-release | grep PRETTY", capture=True)
if result:
    print(f"Pi reachable: {PI_HOST}:{PI_PORT}")
    print(result)
else:
    print(f"Pi unreachable — check PI_HOST/PI_PORT in .env")

_bench["timings"]["ssh_check_s"] = round(time.monotonic() - _t_connect, 1)

## 2. Install Talkbot on Pi

In [ ]:
_t_install = time.monotonic()

install_script = r"""
set -e
if [ -d ~/talkbot ]; then
    echo "talkbot already present — pulling latest"
    cd ~/talkbot && git pull --ff-only
else
    echo "Cloning talkbot..."
    git clone https://github.com/ricklon/talkbot.git ~/talkbot
fi

cd ~/talkbot

if ! command -v uv &>/dev/null; then
    echo "Installing uv..."
    curl -Ls https://astral.sh/uv/install.sh | sh
    export PATH="$HOME/.local/bin:$PATH"
fi

# Install system audio deps (Pi OS)
apt-get install -y --no-install-recommends portaudio19-dev libsndfile1 ffmpeg 2>/dev/null || true

uv sync
echo "talkbot installed OK"
uv run talkbot --version 2>/dev/null || echo "(no --version flag, install OK)"
"""

print(f"Installing talkbot on {PI_HOST}...")
rc = pi_run(install_script, stream=True)
if rc == 0:
    print("\nInstall complete.")
else:
    print(f"\nInstall exited with code {rc} — check output above.")

_bench["timings"]["install_s"] = round(time.monotonic() - _t_install, 1)
print(f"Install took {_bench['timings']['install_s']}s")

## 3. Write Robot Coaching Prompt

In [ ]:
coaching_prompt = """You are a coaching assistant for a SO-ARM101 robot arm.
The student is recording demonstration episodes to train a manipulation policy.

Your role:
- Encourage quality demonstrations (smooth, repeatable motions)
- Track episode count and session progress when told
- Answer questions about the training process in plain language
- Use the coaching metaphor: 'show', 'demonstrate', 'coach' — not 'program'

Keep responses short — the student is physically coaching the robot.
"""

# Write prompt file to Pi
result = pi_run(
    "mkdir -p ~/talkbot/prompts && cat > ~/talkbot/prompts/robot_coach.txt",
    input_data=coaching_prompt
)
if result.returncode == 0:
    print("Coaching prompt written to ~/talkbot/prompts/robot_coach.txt")
else:
    print(f"Write failed: {result.stderr.strip()}")

# Verify
verify = pi_run("cat ~/talkbot/prompts/robot_coach.txt", capture=True)
print()
print(verify)

## 4. Start Talkbot in tmux Session

In [ ]:
# Build env string for talkbot — prompt comes from the file we just wrote
env_parts = [
    f"TALKBOT_LLM_BACKEND={LLM_BACKEND}",
    "TALKBOT_AGENT_PROMPT=$(cat ~/talkbot/prompts/robot_coach.txt)",
]
if LLM_BACKEND == "openrouter" and OPENROUTER_API_KEY:
    env_parts.append(f"OPENROUTER_API_KEY={OPENROUTER_API_KEY}")

env_str = " ".join(env_parts)
start_cmd = f"cd ~/talkbot && export PATH=$HOME/.local/bin:$PATH && {env_str} uv run talkbot"

# Kill any existing session, then start fresh
launch = pi_run(
    f"tmux kill-session -t talkbot 2>/dev/null || true; "
    f"tmux new-session -d -s talkbot '{start_cmd}' && echo launched",
    capture=True
)
if "launched" in launch:
    print(f"Talkbot started in tmux session 'talkbot' on {PI_HOST}")
else:
    print(f"Launch output: {launch}")

print()
print(f"Attach to session:  ssh -p {PI_PORT} {PI_USER}@{PI_HOST}  →  tmux attach -t talkbot")
print(f"Or run locally:     {env_str} uv run talkbot")

## 5. Verify Voice Loop Running

In [ ]:
_t_verify = time.monotonic()

# Give talkbot a moment to start
time.sleep(3)

# Check tmux session exists
sessions = pi_run("tmux list-sessions 2>/dev/null || echo 'no sessions'", capture=True)
print("tmux sessions:")
for line in sessions.splitlines():
    marker = " <<<" if "talkbot" in line else ""
    print(f"  {line}{marker}")

# Grab first few lines of talkbot output
recent = pi_run(
    "tmux capture-pane -pt talkbot -S -20 2>/dev/null || echo 'session not found'",
    capture=True
)
print()
print("Recent talkbot output:")
print(recent[:500] if recent else "(empty)")

running = "talkbot" in sessions and "session not found" not in recent
_bench["talkbot_running"] = running
_bench["timings"]["verify_s"] = round(time.monotonic() - _t_verify, 1)
print()
print(f"Status: {'RUNNING' if running else 'NOT RUNNING — check output above'}")

## Benchmark: Save Timing

In [ ]:
_bench["timings"]["total_s"] = round(time.monotonic() - _t0, 1)
_bench["completed_at"] = datetime.utcnow().isoformat()

results_dir = Path("..") / "bench" / "results"
results_dir.mkdir(exist_ok=True)
ts = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
bench_path = results_dir / f"03_talkbot_{ts}.json"
bench_path.write_text(json.dumps(_bench, indent=2))
print(json.dumps(_bench, indent=2))
print(f"\nBenchmark saved: {bench_path}")